In [ ]:
import pandas as pd
from pathlib import Path
from datasets import Dataset
from transformers import AutoTokenizer
from formatting import processing_function, split_dataset

SYNTAX_DATASET_PATH = "dataset//base//syntax_examples.jsonl"
DOMAIN_DATASET_PATH = "dataset//base//domain_examples.jsonl"
SAVE_FOLDER_PATH = "dataset" 

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048

SYNTAX_DATASET_PATH = Path(SYNTAX_DATASET_PATH).resolve()
DOMAIN_DATASET_PATH = Path(DOMAIN_DATASET_PATH).resolve()
SAVE_FOLDER_PATH = Path(SAVE_FOLDER_PATH).resolve() 

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# NOTE: To avoid recompiling entire dataset, load this cell, then skip to step 8 and just load the data, trim and split it

In [ ]:
# Step 1: Load Syntax and domain examples (incorrect)
df_syntax = pd.read_json(SYNTAX_DATASET_PATH, lines=True)
df_domain = pd.read_json(DOMAIN_DATASET_PATH, lines=True)

df_incorrect = pd.concat([df_syntax, df_domain], ignore_index=True)

before = len(df_incorrect)
df_incorrect = df_incorrect.drop_duplicates()
dropped = before - len(df_incorrect)

print(f"Created incorrect entries dataset. Syntax: {len(df_syntax)}, Domain: {len(df_domain)}, Total: {len(df_incorrect)}")
print(f"({dropped} duplicates found and dropped).")

In [ ]:
# Step 2: Create correct examples
df_correct = df_domain.drop_duplicates(subset="source_id", keep="first").copy()

df_correct["mutation_type"] = "none"
df_correct["mutation_category"] = "none"
df_correct["error_message"] = "none"
df_correct["bad_code"] = df_correct["good_code"]

# Since domain examples are the only ones which involve classification (may or may not be correct)
# we repeat correct examples so as to match the number of domain examples
og_length = len(df_correct)
freq = len(df_domain) / og_length  # float

target_len = int(round(og_length * freq))

full_repeats = target_len // og_length
remainder = target_len % og_length

df_correct = pd.concat(
    [df_correct] * full_repeats + [df_correct.iloc[:remainder]],
    ignore_index=True
)

print(
    f"Created correct entries dataset with freq: {freq:.3f}, "
    f"{og_length} unique entries, total: {len(df_correct)}."
)

In [ ]:
# Step 3: Concenate
df = pd.concat([df_syntax, df_domain, df_correct], ignore_index=True)

print(f"Datasets merged. Final dataset contains {len(df)} entries")

In [ ]:
# Step 4: Add prompts etc, and ids. This can take a while, upto 15 mins, because of tokenizing, patching etc
ds = Dataset.from_pandas(df).remove_columns("__index_level_0__")
ds = ds.map(
    processing_function,
    batched=False,
    fn_kwargs={"tokenizer": tokenizer},
)

def add_ids(example, idx):
    return {"id": idx + 1}

ds = ds.map(add_ids, with_indices=True)
cols = ["id"] + [c for c in ds.column_names if c != "id"]
ds = ds.select_columns(cols)
ds = ds.shuffle(seed=42)

path = Path(SAVE_FOLDER_PATH) / "compiled_dataset.jsonl"
ds.to_json(path, lines=True)

print(f"Dataset saved to {path}")

In [ ]:
# Step 8: Drop longer examples
path = Path(SAVE_FOLDER_PATH) / "compiled_dataset.jsonl"
ds = Dataset.from_pandas(pd.read_json(path, lines=True))
len_before = len(ds)
ds = ds.filter(lambda x: x["length"] <= MAX_SEQ_LENGTH)
len_after = len(ds)

print(f"Dropped {len_before-len_after} entries due to being longer than {MAX_SEQ_LENGTH} tokens.")
print(f"Final dataset size: {len_after} entries.")

In [ ]:
# Step 9: Split datasets and save them
ds_train, ds_eval, ds_test = split_dataset(ds)
    
ds_train.to_json(SAVE_FOLDER_PATH / "split" / "train_dataset.jsonl", lines=True)
ds_eval.to_json(SAVE_FOLDER_PATH / "split" / "eval_dataset.jsonl", lines=True)
ds_test.to_json(SAVE_FOLDER_PATH / "split" / "test_dataset.jsonl", lines=True)

print(f"Split datasets saved to {SAVE_FOLDER_PATH}")

In [ ]:
ds_train = pd.read_json(SAVE_FOLDER_PATH / "split" / "train_dataset.jsonl", lines=True)
ds_eval = pd.read_json(SAVE_FOLDER_PATH / "split" / "eval_dataset.jsonl", lines=True)
ds_test = pd.read_json(SAVE_FOLDER_PATH / "split" / "test_dataset.jsonl", lines=True)

import pandas as pd

def load_and_drop(json_path):
    df = pd.read_json(json_path, lines=True)
    ds = Dataset.from_pandas(df, preserve_index=False)

    keep_cols = [
        "id",
        "source_id",
        "mutation_category",
        "mutation_type",
        "bad_code",
        "error_message",
        "good_code",
        "diff_patch",
    ]

    return ds.select_columns(keep_cols)

ds_train = load_and_drop(SAVE_FOLDER_PATH / "split" / "train_dataset.jsonl")
ds_eval = load_and_drop(SAVE_FOLDER_PATH / "split" / "eval_dataset.jsonl")
ds_test = load_and_drop(SAVE_FOLDER_PATH / "split" / "test_dataset.jsonl")
